# HyPE - Hypothetical Prompt Embeddings

Move the work to index time: embed questions each chunk could answer, not the chunk itself.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

HyDE pays an LLM call per query. **HyPE** generates hypothetical *questions* per chunk once, so query-vs-question matching is question-to-question and queries stay cheap.

In [ ]:
def hypothetical_questions(chunk, n=3):
    data = llm_json(f"Write {n} distinct questions that this passage answers. Reply with a JSON list of strings.\n\nPassage: {chunk}")
    return [x for x in (data or []) if isinstance(x, str)]

print(hypothetical_questions(chunks[0]))

In [ ]:
qs, owner = [], []
for i, chunk in enumerate(chunks):
    for q in hypothetical_questions(chunk):
        qs.append(q)
        owner.append({"chunk": i})
hype_store = build_store(qs, owner)
print(len(qs), "hypothetical questions indexed")

In [ ]:
def hype_search(query, k=3):
    seen, results = set(), []
    for h in hype_store.search(query, k * 3):
        c = h["meta"]["chunk"]
        if c not in seen:
            seen.add(c)
            results.append({"idx": c, "text": chunks[c], "score": h["score"], "meta": {"matched_question": h["text"]}})
    return results[:k]

query = "What does a cross-encoder do?"
for h in hype_search(query):
    print(f"{h['score']:.3f} matched: {h['meta']['matched_question']}")
print(answer(query, [h["text"] for h in hype_search(query)]))